# W-algebras by quantum Hamiltonian reduction

`quantum_hamiltonian_reduction.py` constructs the universal W-algebra $W^k(\mathfrak g, f)$ of Kac–Roan–Wakimoto (Comm. Math. Phys. 241 (2003)) and Kac–Wakimoto (Adv. Math. 185 (2004), with the corrigendum Adv. Math. 193 (2005)) from

* a basic Lie superalgebra $\mathfrak g$: a `LieSuperalgebra` (see `02_lie_superalgebras.ipynb`), whose invariant form, normalised as in Kac–Wakimoto's tables, defines the level;
* an even nilpotent element $f$, and a good grading $x$ for it (by default the Dynkin grading $x = h/2$ of an $\mathfrak{sl}_2$-triple $(e, h, f)$);
* the level $k$: a number, or an element of a field such as $\mathbb Q(k)$.

`WAlgebra(R, g, f, k)` returns a `UniversalVertexAlgebra`, freely generated by fields $J^{\{v\}}$, $v$ in a basis of the centraliser $\mathfrak g^f$, of conformal weight $1 + j$ for $v \in \mathfrak g_{-j}$, with their OPEs and the conformal vector. So everything the library does with universal vertex algebras applies to it: weight spaces, Jacobi identities, ideals and quotients, subalgebras, cosets, Zhu algebras, modules.

**Prerequisites:** `01_getting_started.ipynb`, `02_lie_superalgebras.ipynb`, `03_affine_vertex_algebras.ipynb`.

**Contents**

1. The Virasoro algebra from $\mathfrak{sl}_2$
2. $W_3$ from $\mathfrak{sl}_3$
3. Choosing $f$ and $x$
4. The nilpotent orbits of $\mathfrak{sl}_4$
5. Other Lie algebras
6. Superconformal algebras
7. How it is computed
8. Zhu algebras
9. Reduction of modules
10. The affine method, caching, pickling, limits

In [1]:
import vertex_algebras as va
from vertex_algebras import *
from vertex_algebras import LieSuperalgebra, WAlgebra, UniversalAffineVertexAlgebra
K.<k> = Frac(QQ['k'])

## 1. The Virasoro algebra from $\mathfrak{sl}_2$

The principal nilpotent of $\mathfrak{sl}_2$ gives the universal Virasoro algebra with $c = 1 - 6(k+1)^2/(k+2)$. Without `names=` the generators are called `J{v}` after the basis vectors $v$ of $\mathfrak g^f$; here $\mathfrak g^f$ is spanned by $f$ itself.

In [2]:
sl2 = LieSuperalgebra(QQ, CartanMatrix(['A', 1]))
W = WAlgebra(K, sl2, 'principal', k, conformal_vector='reduction')
print(W)
print(W.degrees(), W.central_charge() == 1 - 6*(k + 1)^2/(k + 2))

Universal vertex algebra on generators ['J{E[-alpha[1]]}'] over Fraction Field of Univariate Polynomial Ring in k over Rational Field
{'J{E[-alpha[1]]}': 2} True


With a name for the generator. The generator is the field $J^{\{f\}}$ of the reduction, which is a multiple of the conformal vector $T$ (both have weight 2, and there is nothing else of weight 2).

In [3]:
Vir = WAlgebra(K, sl2, 'principal', k, names=['L'], conformal_vector='reduction')
L = Vir.gen('L')
T = Vir.conformal_vector()
print('L L ~', L.ope(L))
print('T =', T)
print('T T ~', T.ope(T))

L L ~ [(-k - 2)*dL, (-2*k - 4)*L, 0, -3*k^3 - 23/2*k^2 - 13*k - 4]
T = (-1/(k + 2))*L
T T ~ [(-1/(k + 2))*dL, (-2/(k + 2))*L, 0, (-3*k^2 - 11/2*k - 2)/(k + 2)]


A rational level gives an algebra over $\mathbb Q$. At $k = -2 + 2/5$ it is the universal Virasoro algebra at $c = -22/5$. Its singular vector of weight 4 (killed by $L_1 = T_{(2)}$ and $L_2 = T_{(3)}$, which generate the positive modes) generates the maximal ideal, and the quotient is the Lee–Yang model, with the Rogers–Ramanujan character $\prod_{n \equiv \pm 2 \bmod 5} (1 - q^n)^{-1}$.

In [4]:
LY = WAlgebra(QQ, sl2, 'principal', -2 + 2/5, names=['L'], conformal_vector='reduction')
print(LY.base_ring(), LY.central_charge())


def singular_vectors(V, wt):
    """The vectors of weight wt killed by T_(2) and T_(3) (the modes L_1, L_2)."""
    T = V.conformal_vector()
    B = V.weight_space(wt).basis()
    ims = [[T.nproduct(b, n) for n in (2, 3)] for b in B]
    keys = sorted(set((i, m) for im in ims for i, y in enumerate(im) for m in y.support()), key=str)
    M = matrix(V.base_ring(), [[im[i].monomial_coefficients().get(m, 0) for i, m in keys]
                               for im in ims])
    return [sum(c * b for c, b in zip(row, B)) for row in M.left_kernel().basis()]


v, = singular_vectors(LY, 4)
v

Rational Field -22/5


:L L: + 3/25*d^2L

In [5]:
Q = LY.quotient([v])
print([Q.weight_space(n).dimension() for n in range(13)])
R.<q> = PowerSeriesRing(ZZ)
print(prod(1 / (1 - q^n) for n in range(1, 13) if n % 5 in (2, 3)).add_bigoh(13))

[1, 0, 1, 1, 1, 1, 2, 2, 3, 3, 4, 4, 6]
1 + q^2 + q^3 + q^4 + q^5 + 2*q^6 + 2*q^7 + 3*q^8 + 3*q^9 + 4*q^10 + 4*q^11 + 6*q^12 + O(q^13)


## 2. $W_3$ from $\mathfrak{sl}_3$

The principal reduction of $\mathfrak{sl}_3$ is Zamolodchikov's $W_3$ algebra, $c = 2 - 24(k+2)^2/(k+3)$.

In [6]:
sl3 = LieSuperalgebra(QQ, CartanMatrix(['A', 2]))
W3 = WAlgebra(K, sl3, 'principal', k, names=['L', 'W'], conformal_vector='reduction')
print(W3.degrees(), W3.central_charge() == 2 - 24*(k + 2)^2/(k + 3))
T, Wg = W3.conformal_vector(), W3.gen('W')
print('T =', T)
print('T W ~', T.ope(Wg))

{'L': 2, 'W': 3} True
T = (-1/(k + 3))*L
T W ~ [dW, 3*W, 2*k*L, 0, 24*k^3 + 94*k^2 + 90*k]


The generator of weight 3 comes out as some field of weight 3, not necessarily primary. Adding a multiple of $\partial T$ makes it primary; then its OPE with itself is Zamolodchikov's,
$$W(z)W(w) \sim N\Big(\frac{c/3}{(z-w)^6} + \frac{2T}{(z-w)^4} + \frac{\partial T}{(z-w)^3} + \frac{2\beta\Lambda + \frac{3}{10}\partial^2 T}{(z-w)^2} + \frac{\beta\partial\Lambda + \frac{1}{15}\partial^3 T}{z-w}\Big),$$
with $\Lambda = {:}TT{:} - \frac{3}{10}\partial^2 T$, $\beta = 16/(22 + 5c)$ and a normalisation $N$. The W-algebra satisfies the Jacobi identity (as every W-algebra must; the check is a test of the construction).

In [7]:
t = T.leading_support()
a = -T.nproduct(Wg, 2).monomial_coefficients()[t] / T.nproduct(T.derivative(), 2).monomial_coefficients()[t]
Wp = Wg + a * T.derivative()
print('T_(2) Wp =', T.nproduct(Wp, 2), '  T_(1) Wp = 3 Wp:', T.nproduct(Wp, 1) == 3 * Wp)
c = W3.central_charge()
N = 3 * Wp.nproduct(Wp, 5).monomial_coefficients()[()] / c
beta = 16 / (22 + 5 * c)
Lam = T.no(T) - 3/10 * T.derivative(2)
zam = [N * (beta * Lam.derivative() + T.derivative(3) / 15), N * (2 * beta * Lam + 3/10 * T.derivative(2)),
       N * T.derivative(), 2 * N * T, 0, N * c / 3 * W3.one()]
print('Zamolodchikov:', Wp.ope(Wp) == zam, '  N =', factor(N))
print('Jacobi identity:', W3.jacobi_identity_failures() == {})

T_(2) Wp = 0   T_(1) Wp = 3 Wp: True
Zamolodchikov: True   N = (-5/2) * (k + 4/3) * (k + 12/5) * (k + 3)^2
Jacobi identity: True


## 3. Choosing $f$ and $x$

`f` can be given as

* `'minimal'`: $E_{-\theta}$, $\theta$ the root with $(\theta|\theta) = 2$ of the normalisation of $\mathfrak g$ (Kac–Wakimoto's minimal W-algebras);
* `'principal'`: the principal nilpotent of the even part;
* a Jordan type on the natural representation, for $\mathfrak{sl}_n$, $\mathfrak{so}_n$, $\mathfrak{sp}_n$ (Cartan matrices in Bourbaki's labelling) and $\mathfrak{sl}(m|n)$, $\mathfrak{psl}(m|m)$, $\mathfrak{osp}(m|n)$, $\mathfrak{spo}(n|m)$: a partition, or a pair of partitions for the two parts of $\mathbb C^{m|n}$ (`LieSuperalgebra.nilpotent_of_jordan_type`);
* a list of positive roots $\beta$ ($f = \sum E_{-\beta}$);
* an element of $\mathfrak g$.

The grading $x$ must lie in the Cartan subalgebra, which is why the nilpotents are built as sums of root vectors. In $\mathfrak{sl}_3$ the minimal nilpotent $E_{-\theta}$ and the nilpotent of Jordan type $(2, 1)$, $E_{-\alpha_1}$, are conjugate:

In [8]:
f_min, f_21 = sl3.minimal_nilpotent(), sl3.nilpotent_of_jordan_type([2, 1])
print(f_min, '  ', f_21)
e, h, f = sl3.sl2_triple(f_21)
print('sl_2-triple: e =', e, '  h =', h)
x = sl3.dynkin_grading(f_21)
print('x =', x, ' ', sl3.ad_eigenvalues(x))
print('g^f:', sl3.centralizer_basis(f_21, x))

E[-alpha[1] - alpha[2]]    E[-alpha[1]]
sl_2-triple: e = E[alpha[1]]   h = h1
x = 1/2*h1   {'E[alpha[1]]': 1, 'E[alpha[2]]': -1/2, 'E[alpha[1] + alpha[2]]': 1/2, 'h1': 0, 'h2': 0, 'E[-alpha[1]]': -1, 'E[-alpha[2]]': 1/2, 'E[-alpha[1] - alpha[2]]': -1/2}
g^f: [h1 + 2*h2, E[alpha[2]], E[-alpha[1] - alpha[2]], E[-alpha[1]]]


Both give the Bershadsky–Polyakov algebra, with generators named after different centralisers; the weights, central charges and weight spaces agree.

In [9]:
BP, BP2 = WAlgebra(K, sl3, 'minimal', k, conformal_vector='reduction'), WAlgebra(K, sl3, [2, 1], k, conformal_vector='reduction')
print(BP.degrees())
print(BP2.degrees())
print(BP.central_charge() == BP2.central_charge(), factor(BP.central_charge()))
print([BP.weight_space(n/2).dimension() for n in range(9)])
print([BP2.weight_space(n/2).dimension() for n in range(9)])

{'J{h1 - h2}': 1, 'J{E[-alpha[1]]}': 3/2, 'J{E[-alpha[2]]}': 3/2, 'J{E[-alpha[1] - alpha[2]]}': 2}
{'J{h1 + 2*h2}': 1, 'J{E[alpha[2]]}': 3/2, 'J{E[-alpha[1] - alpha[2]]}': 3/2, 'J{E[-alpha[1]]}': 2}
True (-6) * (k + 3)^-1 * (k + 1/3) * (k + 3/2)
[1, 0, 1, 2, 3, 4, 8, 10, 17]
[1, 0, 1, 2, 3, 4, 8, 10, 17]


The same $f$ in different forms gives the same object:

In [10]:
(WAlgebra(K, sl3, 'principal', k, conformal_vector='reduction') is WAlgebra(K, sl3, [3], k, conformal_vector='reduction') is WAlgebra(K, sl3, [(1, 0), (0, 1)], k, conformal_vector='reduction')
 is WAlgebra(K, sl3, sl3.principal_nilpotent(), k, conformal_vector='reduction'))

True

Any good grading $x$ in the Cartan subalgebra can be given instead of the Dynkin grading. For the minimal nilpotent of $\mathfrak{sl}_3$ there is an even good grading, $\alpha_1(x) = 1$, $\alpha_2(x) = 0$. The conformal vector depends on $x$: here all weights are integers, and the central charge is another one.

In [11]:
xe = (2 * sl3.h(1) + sl3.h(2)) / 3
print(sl3.ad_eigenvalues(xe), sl3.is_good_grading(xe, f_min))
BPe = WAlgebra(K, sl3, 'minimal', k, x=xe, conformal_vector='reduction')
print(BPe.degrees(), factor(BPe.central_charge()))

{'E[alpha[1]]': 1, 'E[alpha[2]]': 0, 'E[alpha[1] + alpha[2]]': 1, 'h1': 0, 'h2': 0, 'E[-alpha[1]]': -1, 'E[-alpha[2]]': 0, 'E[-alpha[1] - alpha[2]]': -1} True
{'J{h1 - h2}': 1, 'J{E[-alpha[2]]}': 1, 'J{E[-alpha[1]]}': 2, 'J{E[-alpha[1] - alpha[2]]}': 2} (-8) * (k + 3)^-1 * (k + 1) * (k + 3/2)


## 4. The nilpotent orbits of $\mathfrak{sl}_4$

The nilpotent orbits of $\mathfrak{sl}_4$ are those of the partitions of 4: $W_4$ (principal, $(4)$), the subregular W-algebra (Feigin–Semikhatov's $W^{(2)}_4$, $(3,1)$), the rectangular one ($(2,2)$) and the minimal one ($(2,1,1)$). Their central charges agree with Kac–Roan–Wakimoto's formula
$$c(\mathfrak g, f, k) = \frac{k\,\mathrm{sdim}\,\mathfrak g}{k + h^\vee} - 12k(x|x) - \sum_{\alpha > 0} (-1)^{p(\alpha)} (12 m_\alpha^2 - 12 m_\alpha + 2) - \tfrac12 \mathrm{sdim}\,\mathfrak g_{1/2},$$
the sum over the roots with $m_\alpha = \alpha(x) > 0$.

In [12]:
def krw_central_charge(g, f, x):
    gr, B = g.ad_eigenvalues(x), g.basis()
    s = sum((-1)^B[a].is_even_odd() * (12*m^2 - 12*m + 2) for a, m in gr.items() if m > 0)
    sh = sum((-1)^B[a].is_even_odd() for a, m in gr.items() if m == 1/2)
    return (k * g.superdimension() / (k + g.dual_coxeter_number()) - 12 * k * g.invariant_form(x, x)
            - s - sh / 2)


sl4 = LieSuperalgebra(QQ, CartanMatrix(['A', 3]))
for lam in Partitions(4):
    if max(lam) == 1:
        continue
    W = WAlgebra(K, sl4, list(lam), k, conformal_vector='reduction')
    f = sl4.nilpotent_of_jordan_type(lam)
    x = sl4.dynkin_grading(f)
    print('%-12s weights %-32s KRW: %s   c = %s' % (lam, sorted(W.degrees().values()),
          W.central_charge() == krw_central_charge(sl4, f, x), factor(W.central_charge())))

[4]          weights [2, 3, 4]                        KRW: True   c = (-60) * (k + 4)^-1 * (k + 11/4) * (k + 16/5)


[3, 1]       weights [1, 2, 2, 2, 3]                  KRW: True   c = (-24) * (k + 4)^-1 * (k + 17/8) * (k + 8/3)
[2, 2]       weights [1, 1, 1, 2, 2, 2, 2]            KRW: True   c = (-12) * (k + 4)^-1 * (k^2 + 41/12*k + 8/3)


[2, 1, 1]    weights [1, 1, 1, 1, 3/2, 3/2, 3/2, 3/2, 2] KRW: True   c = (-6) * (k + 4)^-1 * k * (k + 3/2)


## 5. Other Lie algebras

$\mathfrak{so}_5 \cong \mathfrak{sp}_4$, with long roots corresponding to long roots, so the normalised forms agree and the W-algebras of corresponding orbits have the same weights and central charges: $(5) \leftrightarrow (4)$ (the principal W-algebra $W(2, 4)$), $(3,1,1) \leftrightarrow (2,2)$, $(2,2,1) \leftrightarrow (2,1,1)$ (minimal).

In [13]:
so5, sp4 = LieSuperalgebra(QQ, CartanMatrix(['B', 2])), LieSuperalgebra(QQ, CartanMatrix(['C', 2]))
for l5, l4 in [([5], [4]), ([3, 1, 1], [2, 2]), ([2, 2, 1], [2, 1, 1])]:
    W5, W4 = WAlgebra(K, so5, l5, k, conformal_vector='reduction'), WAlgebra(K, sp4, l4, k, conformal_vector='reduction')
    print('%-10s %-10s weights %-22s same c: %s   c = %s' % (l5, l4, sorted(W5.degrees().values()),
          W5.central_charge() == W4.central_charge(), factor(W5.central_charge())))

[5]        [4]        weights [2, 4]                 same c: True   c = (-60) * (k + 3)^-1 * (k + 13/6) * (k + 12/5)
[3, 1, 1]  [2, 2]     weights [1, 2, 2, 2]           same c: True   c = (-12) * (k + 3)^-1 * (k^2 + 8/3*k + 3/2)
[2, 2, 1]  [2, 1, 1]  weights [1, 1, 1, 3/2, 3/2, 2] same c: True   c = (-6) * (k + 3)^-1 * (k + 1/2) * (k + 1)


Exceptional Lie algebras have no Jordan types here (Bala–Carter labels are not implemented); `'minimal'`, `'principal'`, roots or elements work. The minimal W-algebra of $G_2$:

In [14]:
G2 = LieSuperalgebra(QQ, CartanMatrix(['G', 2]))
WG = WAlgebra(K, G2, 'minimal', k, conformal_vector='reduction')
print(sorted(WG.degrees().values()), factor(WG.central_charge()))
try:
    G2.nilpotent_of_jordan_type([3])
except ValueError as err:
    print('ValueError:', err)

[1, 1, 1, 3/2, 3/2, 3/2, 3/2, 2] (-6) * (k + 4)^-1 * k * (k + 5/3)
ValueError: Jordan types are defined for sl(m|n), psl(m|m), osp(m|n), spo(n|m) and Cartan matrices of type A-D (in Bourbaki's labelling)


## 6. Superconformal algebras

The minimal reductions of Lie superalgebras give the superconformal algebras (Kac–Wakimoto's Table): the Neveu–Schwarz algebra ($N = 1$) from $\mathfrak{spo}(2|1) = \mathfrak{osp}(1|2)$, the $N = 2$ algebra from $\mathfrak{sl}(2|1)$, the $N = 3$ algebra from $\mathfrak{spo}(2|3) = \mathfrak{osp}(3|2)$, the small $N = 4$ algebra from $\mathfrak{psl}(2|2)$, and the big $N = 4$ algebra from $D(2,1;a)$.

In [15]:
for name, g in [('N=1', LieSuperalgebra(QQ, 'spo(2|1)')), ('N=2', LieSuperalgebra(QQ, 'sl(2|1)')),
                ('N=3', LieSuperalgebra(QQ, 'spo(2|3)')), ('small N=4', LieSuperalgebra(QQ, 'psl(2|2)'))]:
    W = WAlgebra(K, g, 'minimal', k, conformal_vector='reduction')
    f = g.minimal_nilpotent()
    print('%-10s %-9s weights %-32s KRW: %s   c = %s' % (name, str(g).split()[2],
          sorted(W.degrees().values()),
          W.central_charge() == krw_central_charge(g, f, g.dynkin_grading(f)), factor(W.central_charge())))

N=1        spo(2|1)  weights [3/2, 2]                         KRW: True   c = (-6) * (k + 3/2)^-1 * (k + 1/2) * (k + 5/4)
N=2        sl(2|1)   weights [1, 3/2, 3/2, 2]                 KRW: True   c = (-6) * (k + 1/2)
N=3        spo(2|3)  weights [1, 1, 1, 3/2, 3/2, 3/2, 2]      KRW: True   c = (-6) * (k + 7/12)


small N=4  psl(2|2)  weights [1, 1, 1, 3/2, 3/2, 3/2, 3/2, 2] KRW: True   c = (-6) * (k + 1)


The big $N = 4$ algebra from $D(2,1;a)$ (over $\mathbb Q(a)$), at a level in $\mathbb Q(a, k)$: six fields of weight 1 ($\mathfrak g^\natural = \mathfrak{sl}_2 \oplus \mathfrak{sl}_2$), four of weight 3/2 and one of weight 2. The central charge $-6k - 3$ does not depend on $a$.

In [16]:
Ka.<a> = Frac(QQ['a'])
Kak = Frac(QQ['a', 'k'])
D = LieSuperalgebra(Ka, 'D(2,1;a)', a=a)
WD = WAlgebra(Kak, D, 'minimal', Kak.gen(1), conformal_vector='reduction')
print(sorted(WD.degrees().values()), factor(WD.central_charge()))

[1, 1, 1, 1, 1, 1, 3/2, 3/2, 3/2, 3/2, 2] (-3) * (2*k + 1)


$N = 1$: with $G$ the field of weight 3/2, $G(z)G(w) \sim \mu\big(\frac{2c/3}{(z-w)^3} + \frac{2T}{z-w}\big)$ for a normalisation $\mu$.

In [17]:
N1 = WAlgebra(K, LieSuperalgebra(QQ, 'spo(2|1)'), 'minimal', k, names=['G', 'L'], conformal_vector='reduction')
G, T = N1.gen('G'), N1.conformal_vector()
t = T.leading_support()
mu = G.nproduct(G, 0).monomial_coefficients()[t] / (2 * T.monomial_coefficients()[t])
print(G.ope(G) == [2 * mu * T, 0, 2 * mu * N1.central_charge() / 3 * N1.one()], '  mu =', mu)
print(factor(N1.central_charge()))

True   mu = -1/2*k - 3/4
(-6) * (k + 3/2)^-1 * (k + 1/2) * (k + 5/4)


$N = 2$: the weight-1 field $J$, normalised so that the two fields of weight 3/2 have charges $\pm 1$, has $J(z)J(w) \sim \frac{c/3}{(z-w)^2}$.

In [18]:
N2 = WAlgebra(K, LieSuperalgebra(QQ, 'sl(2|1)'), 'minimal', k, conformal_vector='reduction')
wts = N2.degrees()
J = N2.gen([s for s, w in wts.items() if w == 1][0])
Gp, Gm = [N2.gen(s) for s, w in wts.items() if w == 3/2]
charge = J.nproduct(Gp, 0).monomial_coefficients()[Gp.leading_support()]
J = J / charge
print('J_(0) G+ = G+:', J.nproduct(Gp, 0) == Gp, '  J_(0) G- = -G-:', J.nproduct(Gm, 0) == -Gm)
print('J J ~', J.ope(J), '  c/3 =', N2.central_charge() / 3)

J_(0) G+ = G+: True   J_(0) G- = -G-: True
J J ~ [0, -2*k - 1]   c/3 = -2*k - 1


Non-minimal nilpotents of Lie superalgebras by Jordan types: $\mathfrak{spo}(2|3) = \mathfrak{osp}(3|2)$ with the principal nilpotent of $\mathfrak{so}_3$ (Jordan types $(1,1)$ on $\mathbb C^2$, $(3)$ on $\mathbb C^3$), and with the principal nilpotent of the even part, $((2), (3))$; $\mathfrak{sl}(3|1)$ with $((3), (1))$.

In [19]:
osp32, sl31 = LieSuperalgebra(QQ, 'spo(2|3)'), LieSuperalgebra(QQ, 'sl(3|1)')
for g, jt in [(osp32, ([1, 1], [3])), (osp32, ([2], [3])), (sl31, ([3], [1]))]:
    W = WAlgebra(K, g, jt, k, conformal_vector='reduction')
    f = g.nilpotent_of_jordan_type(*jt)
    print('%-9s %-14s weights %-28s KRW: %s   c = %s' % (str(g).split()[2], jt,
          sorted(W.degrees().values()),
          W.central_charge() == krw_central_charge(g, f, g.dynkin_grading(f)), factor(W.central_charge())))

spo(2|3)  ([1, 1], [3])  weights [1, 1, 1, 2, 2, 2]           KRW: True   c = (24) * (k + 1/12)
spo(2|3)  ([2], [3])     weights [3/2, 2, 2, 5/2]             KRW: True   c = (18) * (k + 1/3)


sl(3|1)   ([3], [1])     weights [1, 2, 2, 2, 3]              KRW: True   c = (-24) * (k + 2)^-1 * (k + 4/3) * (k + 13/8)


## 7. How it is computed

By Kac–Roan–Wakimoto's Theorem 4.1 (its proof corrected in the corrigendum), $W^k(\mathfrak g, f)$ is the cohomology of $d_{(0)}$ on the complex $C = V^k(\mathfrak g) \otimes F(\text{charged ghosts}) \otimes F(\text{neutral fermions on } \mathfrak g_{1/2})$, and it consists of the $d_{(0)}$-closed elements of charge 0 of the subalgebra $C^-$ (as Kac–Wakimoto remark in the corrigendum). Here $C$ is one universal vertex algebra, $C^-_0$ its subalgebra generated by the dressed currents $J^{(a)}$, $a \in \mathfrak g_{\le 0}$, and the neutral fermions, and $W$ the kernel of $d_{(0)}$ on it, computed weight space by weight space (a `ZeroModeKernel`, a kind of commutant). `V.quantum_hamiltonian_reduction_complex(f)` returns these pieces (`vertex_algebra()`, `differential()`, `subalgebra()`, `kernel()`):

In [20]:
Q = UniversalAffineVertexAlgebra(K, sl2, level=k, conformal_vector='sugawara').quantum_hamiltonian_reduction_complex('principal')
print(Q)
C, d, Cm, Ker = Q.vertex_algebra(), Q.differential(), Q.subalgebra(), Q.kernel()
print(C.variable_names())
print('d =', d, '   d_(0) d =', d.nproduct(d, 0))
print(Ker)
print([Ker.weight_space(n).dimension() for n in range(9)])

Complex of the quantum Hamiltonian reduction of Universal affine vertex algebra of Lie superalgebra with Cartan matrix [[2]] and odd simple roots [] over Rational Field at level k over Fraction Field of Univariate Polynomial Ring in k over Rational Field by E[-alpha[1]]
('J{E[alpha[1]]}', 'J{h1}', 'J{E[-alpha[1]]}', 'phi{E[alpha[1]]}', 'phi*{E[alpha[1]]}')
d = :J{E[alpha[1]]} phi*{E[alpha[1]]}: + phi*{E[alpha[1]]}    d_(0) d = 0
Kernel of the zero modes of [:J{E[alpha[1]]} phi*{E[alpha[1]]}: + phi*{E[alpha[1]]}] on Vertex subalgebra of Universal vertex algebra on generators ['J{E[alpha[1]]}', 'J{h1}', 'J{E[-alpha[1]]}', 'phi{E[alpha[1]]}', 'phi*{E[alpha[1]]}'] over Fraction Field of Univariate Polynomial Ring in k over Rational Field strongly generated by ['S0', 'S1']


[1, 0, 1, 1, 2, 2, 4, 4, 7]


The W-algebra is freely generated (De Sole–Kac): the universal vertex algebra on the generators found has the dimensions of the kernel. For $W_3$:

In [21]:
Ker3 = UniversalAffineVertexAlgebra(K, sl3, level=k, conformal_vector='sugawara').quantum_hamiltonian_reduction_complex('principal').kernel()
print([Ker3.weight_space(n).dimension() for n in range(7)])
print([W3.weight_space(n).dimension() for n in range(7)])

[1, 0, 1, 2, 3, 4, 8]
[1, 0, 1, 2, 3, 4, 8]


The strong generators of the kernel are matched with a basis of $\mathfrak g^f$ by their linear parts. Their OPEs are then computed, by default, through Kac–Wakimoto's Miura map, an injective homomorphism of $W$ into the much smaller $V^{\tau}(\mathfrak g_0) \otimes F(\mathfrak g_{1/2})$ (`algorithm='miura'`); `algorithm='complex'` computes them in $C$. The result is the same algebra:

In [22]:
W3c = WAlgebra(K, sl3, 'principal', k, names=['L', 'W'], algorithm='complex', conformal_vector='reduction')
print(W3c is W3)

True


## 8. Zhu algebras

De Sole–Kac (*Finite vs affine W-algebras*) identify the finite W-algebra $U(\mathfrak g, f)$ with the **H-twisted** Zhu algebra of $W^k(\mathfrak g, f)$: Zhu's algebra of the modules twisted by $e^{2\pi i L_0}$, on which the fields of half-integer weight have zero modes too (`zhu_algebra(twist='H')`, see `09_zhu_algebras.ipynb`, section 10). With integer weights the twist is trivial. For $W_3$ it is the centre of $U(\mathfrak{sl}_3)$ (Kostant), a polynomial algebra in two generators: all commutators vanish.

In [23]:
A = W3.zhu_algebra()
xL, xW = A.gens()
print(A, A.is_known_exact())
print('commutative:', xL * xW == xW * xL, '  nonzero commutators:', A.relations())

Zhu algebra of Universal vertex algebra on generators ['L', 'W'] over Fraction Field of Univariate Polynomial Ring in k over Rational Field True
commutative: True   nonzero commutators: {}


With the even good grading of section 3 the Bershadsky–Polyakov algebra has integer weights, so the Zhu algebra computed here is exact (`is_known_exact()`): a non-commutative algebra on the classes of the four generators, with these commutators (`relations()`):

In [24]:
Ae = BPe.zhu_algebra()
print(Ae, Ae.is_known_exact())
for (p, q), r in Ae.relations().items():
    print('[[%s], [%s]] = %s' % (p, q, r))

Zhu algebra of Universal vertex algebra on generators ['J{h1 - h2}', 'J{E[-alpha[2]]}', 'J{E[-alpha[1]]}', 'J{E[-alpha[1] - alpha[2]]}'] over Fraction Field of Univariate Polynomial Ring in k over Rational Field True
[[J{E[-alpha[2]]}], [J{h1 - h2}]] = -3*[J{E[-alpha[2]]}]
[[J{E[-alpha[1]]}], [J{h1 - h2}]] = 3*[J{E[-alpha[1]]}]
[[J{E[-alpha[1]]}], [J{E[-alpha[2]]}]] = (-1/3*k-1)*[J{h1 - h2}] + 1/3*[J{h1 - h2}]^2 + [J{E[-alpha[1] - alpha[2]]}]
[[J{E[-alpha[1] - alpha[2]]}], [J{E[-alpha[2]]}]] = -k*[J{E[-alpha[2]]}]
[[J{E[-alpha[1] - alpha[2]]}], [J{E[-alpha[1]]}]] = k*[J{E[-alpha[1]]}]


With the Dynkin grading the Bershadsky–Polyakov algebra has the generators $G^\pm$ of weight 3/2. The untwisted Zhu algebra sees only $[J]$ and $[L]$ (and is not claimed exact); the H-twisted one has all four generators and is exact, the finite W-algebra of the minimal nilpotent of $\mathfrak{sl}_3$. $[J]$ acts on $[G^\pm]$ by $\pm 3$, the commutator of $[G^\pm]$ is quadratic in $[J]$ plus $[L]$:

In [25]:
print(BP.zhu_algebra())
AB = BP.zhu_algebra(twist='H')
print(AB, AB.is_known_exact())
for (p, q), r in AB.relations().items():
    print('[[%s], [%s]] = %s' % (p, q, r))

Algebra mapping onto the Zhu algebra of Universal vertex algebra on generators ['J{h1 - h2}', 'J{E[-alpha[1]]}', 'J{E[-alpha[2]]}', 'J{E[-alpha[1] - alpha[2]]}'] over Fraction Field of Univariate Polynomial Ring in k over Rational Field
H-twisted Zhu algebra of Universal vertex algebra on generators ['J{h1 - h2}', 'J{E[-alpha[1]]}', 'J{E[-alpha[2]]}', 'J{E[-alpha[1] - alpha[2]]}'] over Fraction Field of Univariate Polynomial Ring in k over Rational Field True
[[J{E[-alpha[1]]}], [J{h1 - h2}]] = 3*[J{E[-alpha[1]]}]
[[J{E[-alpha[2]]}], [J{h1 - h2}]] = -3*[J{E[-alpha[2]]}]
[[J{E[-alpha[2]]}], [J{E[-alpha[1]]}]] = (1/4*k^2+5/8*k+3/8) + (1/6*k+1/2)*[J{h1 - h2}] - 1/3*[J{h1 - h2}]^2 - [J{E[-alpha[1] - alpha[2]]}]
[[J{E[-alpha[1] - alpha[2]]}], [J{E[-alpha[1]]}]] = (-1/2*k-3/2)*[J{E[-alpha[1]]}]
[[J{E[-alpha[1] - alpha[2]]}], [J{E[-alpha[2]]}]] = (1/2*k+3/2)*[J{E[-alpha[2]]}]


Its centre in filtration degree $\le 3$ is spanned by $1$ and two Casimirs, of degrees 2 and 3, as the centre of $U(\mathfrak{sl}_3)$, which is the centre of every finite W-algebra of $\mathfrak{sl}_3$ (Premet):

In [26]:
from itertools import combinations_with_replacement
mons = [m for d in range(4) for m in combinations_with_replacement(range(4), d)
        if AB.degree_on_basis(m) <= 3]
els = [AB.monomial(m) for m in mons]
rows = []
for x in AB.gens():
    ims = [(x * e - e * x).monomial_coefficients() for e in els]
    keys = sorted(set(kk for d in ims for kk in d))
    rows += [[d.get(kk, 0) for d in ims] for kk in keys]
print('central elements of degree <= 3:', matrix(K, rows).right_kernel_matrix().nrows())

central elements of degree <= 3: 3


## 9. Reduction of modules

The reduction is a functor: a module $M$ of $V^k(\mathfrak g)$ gives a module $H(M)$ of $W^k(\mathfrak g, f)$. `quantum_hamiltonian_reduction_modules.py` implements it for highest weight modules, a Verma module $M(\lambda)$ (induced from a Verma module of the Zhu algebra) or its simple quotient $L(\lambda)$, with `M.quantum_hamiltonian_reduction(f, reduction=...)`. The pieces of the complex $M \otimes F^{ch} \otimes F^{ne}$ are infinite dimensional (Arakawa), so $H(M)$ is not computed as cohomology: the theorems give it, and their hypotheses are checked.

* `reduction='+'`, the differential the W-algebras above are built with: $H(M(\lambda))$ is the Verma module of $W$ of the highest weight below, for any good $(x, f)$ (Kac–Wakimoto 2004, Thm 6.3); $H(L(\lambda))$ for minimal $f$ is $0$ if $k - (\lambda|\theta) \in \mathbb Z_{\ge 0}$, else the simple quotient (Arakawa, Duke 2005); for principal $f$ under Arakawa's genericity condition (Invent. 2007, Thm 9.1.4).
* `reduction='-'`, principal $f$ of a Lie algebra (Arakawa 2007): $H(L(\lambda))$ is nonzero iff $\lambda$ is anti-dominant.

The highest weight of $W$ (the eigenvalues of the zero modes of its generators) comes from the Miura map. An $\mathfrak{sl}_2$ Verma module with symbolic level and weight reduces to the Virasoro Verma module with $h = \frac{\lambda(\lambda+2)}{4(k+2)} - \frac{\lambda}{2}$:

In [27]:
Kl.<kk, lam> = Frac(QQ['kk', 'lam'])
V2 = UniversalAffineVertexAlgebra(Kl, ['A', 1], level=kk)
M = V2.induced_module(V2.zhu_algebra().verma_module({'alphacheck[1]': lam}))
H = M.quantum_hamiltonian_reduction('principal')
print(H)
print(H.top_weight() == lam*(lam + 2)/(4*(kk + 2)) - lam/2, H.vertex_algebra().central_charge())

Verma module of highest weight {'J{E[-alpha[1]]}': 1/2*kk*lam - 1/4*lam^2 + 1/2*lam} over Universal vertex algebra on generators ['J{E[-alpha[1]]}'] over Fraction Field of Multivariate Polynomial Ring in kk, lam over Rational Field
True (-6*kk^2 - 11*kk - 4)/(kk + 2)


The highest weight alone, without building modules: for $W_3$ the eigenvalues of $L_0$ and $W_0$ on the reduction of $M(\lambda)$, $\lambda = (a, b)$ in Dynkin labels. $L_0$ is Kac–Wakimoto's $\frac{(\lambda|\lambda+2\rho)}{2(k+3)} - \lambda(x)$:

In [28]:
K3.<k3, a, b> = Frac(QQ['k3', 'a', 'b'])
V3 = UniversalAffineVertexAlgebra(K3, ['A', 2], level=k3)
values, h = V3.quantum_hamiltonian_reduction_highest_weight('principal', [a, b], names=['L', 'W'])
print(values)
print(h == (2*(a^2 + a*b + b^2)/3 + 2*a + 2*b)/(2*(k3 + 3)) - (a + b))

{'L': k3*a - 1/3*a^2 + k3*b - 1/3*a*b - 1/3*b^2 + 2*a + 2*b, 'W': -4/3*k3^2*a + 2/3*k3*a^2 - 2/27*a^3 - 2/3*k3^2*b + 1/3*k3*a*b - 1/9*a^2*b + 1/9*a*b^2 + 2/27*b^3 - 10/3*k3*a + 2/3*a^2 - 2/3*k3*b - 2/3*b^2 - 4/3*a + 4/3*b}
True


**The Lee–Yang model from $\mathfrak{sl}_2$.** At $k = -2 + 2/5$ the admissible weights are $\lambda = 0, -2/5, -4/5, -6/5, -8/5$. Both reductions of the simple modules $L(\lambda)$ give exactly the two Lee–Yang modules ($h = 0, -1/5$), each twice, and one zero module, with the Rogers–Ramanujan characters:

In [29]:
from vertex_algebras.vertex_algebra_submodule import ZeroModule
VL = UniversalAffineVertexAlgebra(QQ, ['A', 1], level=-2 + 2/5)
AL = VL.zhu_algebra()


def simple(V, weight):
    return V.induced_module(V.zhu_algebra().verma_module(weight)).simple_quotient()


for l in [0, -2/5, -4/5, -6/5, -8/5]:
    L = simple(VL, {'alphacheck[1]': l})
    out = []
    for red in ('+', '-'):
        H = L.quantum_hamiltonian_reduction('principal', reduction=red)
        out.append('0' if isinstance(H, ZeroModule) else 'L(%s)' % H.top_weight())
    print('lambda = %5s:  H_+ = %-8s H_- = %s' % (l, out[0], out[1]))
H = simple(VL, {'alphacheck[1]': -4/5}).quantum_hamiltonian_reduction('principal')
print(H.vertex_algebra().central_charge(), H.character(6))

lambda =     0:  H_+ = L(0)     H_- = 0
lambda =  -2/5:  H_+ = L(-1/5)  H_- = L(0)
lambda =  -4/5:  H_+ = L(-1/5)  H_- = L(-1/5)
lambda =  -6/5:  H_+ = L(0)     H_- = L(-1/5)


lambda =  -8/5:  H_+ = 0        H_- = L(0)
-22/5 q^(-1/5) + q^(4/5) + q^(9/5) + q^(14/5) + 2*q^(19/5) + 2*q^(24/5) + 3*q^(29/5) + O(q^6)


**The 3-state Potts model from $\mathfrak{sl}_3$.** At $k + 3 = 4/5$ the reduction is $W_3$ at $c = 4/5$. The "−" reduction of the simple modules $L(\lambda^I - (k+3)\lambda^F)$, $\lambda^I$ integrable of level 1 and $\lambda^F$ dominant of level $\le 4$, gives zero or one of the $W_3$ minimal model modules, of conformal weights $0, 1/15, 2/5, 2/3$:

In [30]:
VP = UniversalAffineVertexAlgebra(QQ, ['A', 2], level=-3 + 4/5)
tops = {}
for l0 in [(0, 0), (1, 0), (0, 1)]:
    for m1 in range(5):
        for m2 in range(5 - m1):
            lam_ = {'alphacheck[1]': l0[0] - 4/5*m1, 'alphacheck[2]': l0[1] - 4/5*m2}
            H = simple(VP, lam_).quantum_hamiltonian_reduction('principal', reduction='-', names=['L', 'W'])
            if not isinstance(H, ZeroModule):
                tops[H.top_weight()] = tops.get(H.top_weight(), 0) + 1
print(H.vertex_algebra().central_charge(), sorted(tops.items()))

4/5 [(0, 3), (1/15, 6), (2/5, 3), (2/3, 6)]


**Minimal reductions.** For the Bershadsky–Polyakov algebra ($\mathfrak{sl}_3$, minimal) at $k = -3/2$, $H(L(\lambda)) = 0$ exactly when $k - (\lambda|\theta) = k - \lambda_1 - \lambda_2$ is a non-negative integer:

In [31]:
VB = UniversalAffineVertexAlgebra(QQ, ['A', 2], level=-3/2)
for l in [(-1/2, -1/2), (-1/2, -1), (0, -3/2), (0, 0), (1/2, -2)]:
    H = simple(VB, {'alphacheck[1]': l[0], 'alphacheck[2]': l[1]}).quantum_hamiltonian_reduction('minimal')
    print(l, -3/2 - l[0] - l[1], H if isinstance(H, ZeroModule) else 'top weight %s' % H.top_weight())

(-1/2, -1/2) -1/2 top weight 0
(-1/2, -1) 0 Zero module over Universal vertex algebra on generators ['J{h1 - h2}', 'J{E[-alpha[1]]}', 'J{E[-alpha[2]]}', 'J{E[-alpha[1] - alpha[2]]}'] over Rational Field
(0, -3/2) 0 Zero module over Universal vertex algebra on generators ['J{h1 - h2}', 'J{E[-alpha[1]]}', 'J{E[-alpha[2]]}', 'J{E[-alpha[1] - alpha[2]]}'] over Rational Field
(0, 0) -3/2 top weight 0
(1/2, -2) 0 Zero module over Universal vertex algebra on generators ['J{h1 - h2}', 'J{E[-alpha[1]]}', 'J{E[-alpha[2]]}', 'J{E[-alpha[1] - alpha[2]]}'] over Rational Field


The Euler–Poincaré character of Kac–Roan–Wakimoto (Thm 3.1) of the reduction of a Verma module is the character of the Verma module of $W$; here for $N = 2$ (from $\mathfrak{sl}(2|1)$), where the creation modes $G^\pm_{-1/2}$ contribute $q^{1/2}$:

In [32]:
VN = UniversalAffineVertexAlgebra(QQ, LieSuperalgebra(QQ, 'sl(2|1)'), level=1/3)
print(VN.quantum_hamiltonian_reduction_character('minimal', [1/2, 1/5], 3))
WN = VN.quantum_hamiltonian_reduction('minimal', conformal_vector='reduction')
print(WN.verma_module(VN.quantum_hamiltonian_reduction_highest_weight('minimal', [1/2, 1/5])[0]).character(3))

q^(-71/200) + 2*q^(29/200) + 3*q^(129/200) + 6*q^(229/200) + 11*q^(329/200) + 18*q^(429/200) + 28*q^(529/200) + O(q^3)
q^(-71/200) + 2*q^(29/200) + 3*q^(129/200) + 6*q^(229/200) + 11*q^(329/200) + 18*q^(429/200) + 28*q^(529/200) + O(q^3)


Not yet: the cohomology itself (a finite model is needed), the "−" reduction for other $f$ (it gives Ramond-twisted modules), W-algebras whose Verma modules have infinite top levels (non-abelian $\mathfrak g^\natural$, e.g. the minimal reduction of $\mathfrak{sl}_4$), Weyl modules, and simple modules for other $f$ with "+"; these are refused:

In [33]:
V4 = UniversalAffineVertexAlgebra(QQ, ['A', 3], level=1/2)
M4 = V4.induced_module(V4.zhu_algebra().verma_module({'alphacheck[%d]' % i: 0 for i in (1, 2, 3)}))
try:
    M4.quantum_hamiltonian_reduction('minimal')
except NotImplementedError as e:
    print(e)

the Verma modules of Universal vertex algebra on generators ['J{E[alpha[2]]}', 'J{h1 - h3}', 'J{h2}', 'J{E[-alpha[2]]}', 'J{E[-alpha[1]]}', 'J{E[-alpha[3]]}', 'J{E[-alpha[1] - alpha[2]]}', 'J{E[-alpha[2] - alpha[3]]}', 'J{E[-alpha[1] - alpha[2] - alpha[3]]}'] over Rational Field have infinite top levels (a generator of integer weight with nonzero weight under h ∩ g^f): not implemented yet


## 10. The affine method, caching, pickling, limits

The universal affine vertex algebra of a `LieSuperalgebra` or of a Cartan type has the reduction as a method, with its level:

In [34]:
V = UniversalAffineVertexAlgebra(K, ['A', 2], level=k, conformal_vector='sugawara')
print(V.quantum_hamiltonian_reduction('principal', names=['L', 'W'], conformal_vector='reduction') is W3)
V2 = UniversalAffineVertexAlgebra(K, LieSuperalgebra(QQ, 'sl(2|1)'), level=k, conformal_vector='sugawara')
print(V2.quantum_hamiltonian_reduction('minimal', conformal_vector='reduction') is N2)
print(loads(dumps(W3)) is W3)

True


True
True


Limits:

* The grading $x$ must lie in the Cartan subalgebra. A nilpotent whose $\mathfrak{sl}_2$-triples all have $h$ outside it is refused; give a conjugate sum of root vectors (a Jordan type) instead.
* Jordan types are available for the classical types only.
* The computation of the kernel dominates the time: $W_4$ takes about a second, $W(G_2)$ about 20 s, $W_5$ about 100 s and $W(B_3)$ about 250 s.

In [35]:
f = sl3.root_vector((1, 0), -1) + sl3.root_vector((1, 1), -1)
try:
    WAlgebra(K, sl3, f, k, conformal_vector='reduction')
except NotImplementedError as err:
    print('NotImplementedError:', err)

NotImplementedError: the grading x must lie in the Cartan subalgebra: give f as a sum of root vectors, or x


## References

- [KRW2003] V. G. Kac, S.-S. Roan, M. Wakimoto, *Quantum reduction for affine superalgebras*, Comm. Math. Phys. 241 (2003), 307–342, [arXiv:math-ph/0302015](https://arxiv.org/abs/math-ph/0302015).
- [KW2004] V. G. Kac, M. Wakimoto, *Quantum reduction and representation theory of superconformal algebras*, Adv. Math. 185 (2004), 400–458; corrigendum, Adv. Math. 193 (2005), 453–455, [arXiv:math-ph/0304011](https://arxiv.org/abs/math-ph/0304011). — the quantum Hamiltonian reduction, the Miura map, the reduction of Verma modules (Thm 6.3).
- [Zam1985] A. B. Zamolodchikov, *Infinite additional symmetries in two-dimensional conformal quantum field theory*, Theoret. and Math. Phys. 65 (1985), 1205–1213, [doi:10.1007/BF01036128](https://doi.org/10.1007/BF01036128). — the $W_3$ algebra (section 2).
- [Ber1991] M. Bershadsky, *Conformal field theories via Hamiltonian reduction*, Comm. Math. Phys. 139 (1991), 71–82, [doi:10.1007/BF02102729](https://doi.org/10.1007/BF02102729).
- [Pol1990] A. M. Polyakov, *Gauge transformations and diffeomorphisms*, Internat. J. Modern Phys. A 5 (1990), 833–842, [doi:10.1142/S0217751X90000386](https://doi.org/10.1142/S0217751X90000386). — the Bershadsky–Polyakov algebra (section 3).
- [FS2004] B. L. Feigin, A. M. Semikhatov, *W⁽²⁾ₙ algebras*, Nuclear Phys. B 698 (2004), 409–449, [doi:10.1016/j.nuclphysb.2004.06.056](https://doi.org/10.1016/j.nuclphysb.2004.06.056). — the subregular W-algebra $W^{(2)}_4$ (section 4).
- [DSK2005] A. De Sole, V. G. Kac, *Freely generated vertex algebras and non-linear Lie conformal algebras*, Comm. Math. Phys. 254 (2005), 659–694, [arXiv:math-ph/0312042](https://arxiv.org/abs/math-ph/0312042). — free generation (section 7).
- [DSK2006] A. De Sole, V. G. Kac, *Finite vs affine W-algebras*, Jpn. J. Math. 1 (2006), 137–261, [arXiv:math-ph/0511055](https://arxiv.org/abs/math-ph/0511055). — finite W-algebras as H-twisted Zhu algebras (section 8).
- [Ara2005] T. Arakawa, *Representation theory of superconformal algebras and the Kac–Roan–Wakimoto conjecture*, Duke Math. J. 130 (2005), 435–478, [arXiv:math-ph/0405015](https://arxiv.org/abs/math-ph/0405015).
- [Ara2007] T. Arakawa, *Representation theory of W-algebras*, Invent. Math. 169 (2007), 219–320, [arXiv:math/0506056](https://arxiv.org/abs/math/0506056). — the reduction of highest weight modules (section 9).